# Lab 02 Extra - Banco de Dados Northwind
**Disciplina:** Extração e Preparação de Dados | **Professor:** Luis Aramis

Este é um notebook extra para praticar SQL com um banco de dados clássico: o **Northwind**.
Ele simula uma importadora/exportadora de alimentos gourmet.

## 1. Setup e Download
Vamos baixar o `northwind.db` e conectar o SQLAlchemy.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
import urllib.request

# Download do northwind.db
if not os.path.exists('northwind.db'):
    # URL do repositório jpwhite3/northwind-SQLite3
    url = 'https://github.com/jpwhite3/northwind-SQLite3/raw/main/dist/northwind.db'
    urllib.request.urlretrieve(url, 'northwind.db')
    print('Banco Northwind baixado com sucesso!')

engine = create_engine('sqlite:///northwind.db')
print('Conexão estabelecida!')

Banco Northwind baixado com sucesso!
Conexão estabelecida!


## 2. Mapa do Banco
Quais tabelas temos aqui?

In [2]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tabelas = pd.read_sql(query, engine)
print(tabelas)

                    name
0             Categories
1        sqlite_sequence
2   CustomerCustomerDemo
3   CustomerDemographics
4              Customers
5              Employees
6    EmployeeTerritories
7          Order Details
8                 Orders
9               Products
10               Regions
11              Shippers
12             Suppliers
13           Territories


## 3. Consultas Básicas
1. Liste os 5 produtos mais caros (`Products`).
2. Liste todos os clientes (`Customers`) que moram no 'Brazil'.

In [3]:
query_produtos_caros = """
SELECT ProductName, UnitPrice
FROM Products
ORDER BY UnitPrice DESC
LIMIT 5;
"""

produtos_caros = pd.read_sql(query_produtos_caros, engine)
print(produtos_caros)


               ProductName  UnitPrice
0            Côte de Blaye     263.50
1  Thüringer Rostbratwurst     123.79
2          Mishi Kobe Niku      97.00
3   Sir Rodney's Marmalade      81.00
4         Carnarvon Tigers      62.50


In [4]:
query_clientes_brasil = """
SELECT CustomerID, CompanyName, City, Country
FROM Customers
WHERE Country = 'Brazil';
"""

clientes_brasil = pd.read_sql(query_clientes_brasil, engine)
print(clientes_brasil)


  CustomerID             CompanyName            City Country
0      COMMI        Comércio Mineiro       Sao Paulo  Brazil
1      FAMIA      Familia Arquibaldo       Sao Paulo  Brazil
2      GOURL     Gourmet Lanchonetes        Campinas  Brazil
3      HANAR           Hanari Carnes  Rio de Janeiro  Brazil
4      QUEDE             Que Delícia  Rio de Janeiro  Brazil
5      QUEEN           Queen Cozinha       Sao Paulo  Brazil
6      RICAR      Ricardo Adocicados  Rio de Janeiro  Brazil
7      TRADH  Tradição Hipermercados       Sao Paulo  Brazil
8      WELLI  Wellington Importadora         Resende  Brazil


## 4. JOIN: Pedidos e Clientes
Vamos ver quem fez quais pedidos.
Tabelas: `Orders` e `Customers`.
Chave de ligação: `CustomerID`.

In [6]:
query_join = """
SELECT 
    o.OrderID,
    o.OrderDate,
    c.CustomerID,
    c.CompanyName,
    c.Country
FROM Orders o
JOIN Customers c 
    ON o.CustomerID = c.CustomerID
ORDER BY o.OrderDate;
"""

df_join = pd.read_sql(query_join, engine)
print(df_join)

       OrderID            OrderDate CustomerID                CompanyName  \
0        18429  2012-07-10 15:40:46      GREAL    Great Lakes Food Market   
1        25506  2012-07-10 20:28:57      RICAR         Ricardo Adocicados   
2        26048  2012-07-11 01:09:16      LONEP   Lonesome Pine Restaurant   
3        16958  2012-07-11 20:26:28      FOLKO             Folk och fä HB   
4        25877  2012-07-11 21:17:36      NORTS                North/South   
...        ...                  ...        ...                        ...   
16277    11298  2023-10-25 13:00:29      WHITC       White Clover Markets   
16278    23676  2023-10-26 06:28:53      ROMEY           Romero y tomillo   
16279    13789  2023-10-27 06:38:44      MORGK     Morgenstern Gesundkost   
16280    25677  2023-10-27 18:17:38      BOLID  Bólido Comidas preparadas   
16281    13724  2023-10-28 00:09:48      MAISD               Maison Dewey   

       Country  
0          USA  
1       Brazil  
2          USA  
3      

## 5. JOIN Triplo: Detalhes do Pedido
O que tem dentro do pedido 10248?
Caminho: `OrderDetails` -> `Products`.

In [8]:
query_pedido = """
SELECT 
    od.OrderID,
    p.ProductName,
    od.Quantity,
    od.UnitPrice,
    (od.Quantity * od.UnitPrice) AS Total
FROM "Order Details" od
JOIN Products p
    ON od.ProductID = p.ProductID
WHERE od.OrderID = 10248;
"""

df_pedido = pd.read_sql(query_pedido, engine)
print(df_pedido)

   OrderID                    ProductName  Quantity  UnitPrice  Total
0    10248                 Queso Cabrales        12       14.0  168.0
1    10248  Singaporean Hokkien Fried Mee        10        9.8   98.0
2    10248         Mozzarella di Giovanni         5       34.8  174.0


## 6. Desafio: Total de Vendas por Categoria
Descubra qual Categoria de produtos (`Categories`) gerou mais receita.
Dica: Você vai precisar ligar `Categories` -> `Products` -> `Order Details`.

In [9]:
query_categoria = """
SELECT 
    c.CategoryName,
    SUM(od.UnitPrice * od.Quantity) AS ReceitaTotal
FROM Categories c
JOIN Products p
    ON c.CategoryID = p.CategoryID
JOIN "Order Details" od
    ON p.ProductID = od.ProductID
GROUP BY c.CategoryID, c.CategoryName
ORDER BY ReceitaTotal DESC;
"""

df_categoria = pd.read_sql(query_categoria, engine)
print(df_categoria)

     CategoryName  ReceitaTotal
0       Beverages   92181842.95
1     Confections   66347544.94
2    Meat/Poultry   64896314.41
3  Dairy Products   58034940.00
4      Condiments   55802774.45
5         Seafood   49931965.52
6         Produce   32706403.90
7  Grains/Cereals   28573512.55
